# External validation of the distress criterion

This notebook checks whether the V4FinBench distress criterion is associated with independently reported bankruptcy outcomes. The criterion is applied without fitting or threshold selection:

$$
\frac{\mathrm{equity}}{\mathrm{total\ assets}} < 0
\quad\land\quad
\frac{\mathrm{EBITDA}}{\mathrm{total\ assets}} < 0
\quad\land\quad
\frac{\mathrm{current\ assets}}{\mathrm{current\ liabilities}} < 0.6.
$$

Two external datasets are used:

1. Polish Companies Bankruptcy (UCI dataset 365), with reported bankruptcy outcomes one to five years ahead.
2. American Companies Bankruptcy, with Chapter 7 or Chapter 11 filings and a one-year-ahead event construction.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import arff
from scipy.stats import fisher_exact


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


ROOT = find_project_root(Path.cwd().resolve())
POLISH_DIR = ROOT / "data" / "polish_companies_bankruptcy"
AMERICAN_FILE = (
    ROOT / "data" / "american_companies_bankruptcy" / "american_bankruptcy.csv"
)

CURRENT_RATIO_THRESHOLD = 0.6

## Metrics

Precision answers how often a criterion-positive company enters bankruptcy. Recall measures how many bankruptcy events the deliberately narrow criterion captures. Lift compares precision with the unconditional bankruptcy rate. The odds ratio and Fisher exact test quantify association; the Matthews correlation coefficient summarizes the complete confusion matrix.

In [ ]:
def safe_ratio(numerator: float, denominator: float) -> float:
    return numerator / denominator if denominator else np.nan


def binary_metrics(y_true: pd.Series, criterion: pd.Series) -> dict[str, float]:
    y_true = y_true.astype(bool).to_numpy()
    criterion = criterion.astype(bool).to_numpy()

    tp = int(np.sum(criterion & y_true))
    fp = int(np.sum(criterion & ~y_true))
    fn = int(np.sum(~criterion & y_true))
    tn = int(np.sum(~criterion & ~y_true))

    prevalence = safe_ratio(tp + fn, len(y_true))
    precision = safe_ratio(tp, tp + fp)
    recall = safe_ratio(tp, tp + fn)
    specificity = safe_ratio(tn, tn + fp)
    lift = safe_ratio(precision, prevalence)
    odds_ratio, fisher_p = fisher_exact([[tp, fp], [fn, tn]])

    mcc_denominator = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = safe_ratio(tp * tn - fp * fn, mcc_denominator)

    return {
        "observations": len(y_true),
        "bankruptcies": tp + fn,
        "criterion_positive": tp + fp,
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn,
        "true_negative": tn,
        "prevalence": prevalence,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "lift": lift,
        "odds_ratio": float(odds_ratio),
        "fisher_p": float(fisher_p),
        "mcc": mcc,
    }


PERCENT_COLUMNS = ["prevalence", "precision", "recall", "specificity"]


def display_results(results: pd.DataFrame):
    formats = {column: "{:.2%}" for column in PERCENT_COLUMNS}
    formats.update(
        {
            "lift": "{:.2f}",
            "odds_ratio": "{:.2f}",
            "fisher_p": "{:.2e}",
            "mcc": "{:.3f}",
        }
    )
    return results.style.format(formats)

## Polish Companies Bankruptcy

The required variables are available directly: `Attr10` is equity/total assets, `Attr48` is EBITDA/total assets, and `Attr4` is the current ratio. Rows missing any of these variables are excluded. The source files use a reversed naming convention: `5year.arff` contains observations one year before the outcome.

In [ ]:
def load_polish_task(path: Path) -> pd.DataFrame:
    records, _ = arff.loadarff(path)
    frame = pd.DataFrame(records)
    frame["class"] = frame["class"].map({b"0": 0, b"1": 1})
    return frame


def polish_criterion(
    frame: pd.DataFrame,
    current_ratio_threshold: float = CURRENT_RATIO_THRESHOLD,
) -> pd.Series:
    return (
        frame["Attr10"].lt(0)
        & frame["Attr48"].lt(0)
        & frame["Attr4"].lt(current_ratio_threshold)
    )


polish_rows = []
polish_tasks = {}

for path in sorted(POLISH_DIR.glob("*year.arff")):
    source_year = int(path.stem.removesuffix("year"))
    horizon = 6 - source_year
    frame = load_polish_task(path)
    complete = frame.dropna(subset=["Attr4", "Attr10", "Attr48"]).copy()
    polish_tasks[horizon] = complete

    row = {"horizon": horizon, "excluded_missing": len(frame) - len(complete)}
    row.update(binary_metrics(complete["class"], polish_criterion(complete)))
    polish_rows.append(row)

polish_results = pd.DataFrame(polish_rows).sort_values("horizon")
display_results(polish_results.set_index("horizon"))

### Liquidity-threshold sensitivity

The equity and EBITDA cutoffs remain fixed at zero. Only the current-ratio cutoff is varied. This checks whether the result depends narrowly on the selected value of 0.6.

In [ ]:
one_year_polish = polish_tasks[1]
threshold_rows = []

for threshold in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2):
    row = {"current_ratio_threshold": threshold}
    row.update(
        binary_metrics(
            one_year_polish["class"],
            polish_criterion(one_year_polish, threshold),
        )
    )
    threshold_rows.append(row)

threshold_results = pd.DataFrame(threshold_rows)
display_results(threshold_results.set_index("current_ratio_threshold"))

## American Companies Bankruptcy

The accounting variables are reported as levels. Equity is reconstructed as total assets minus total liabilities. Lombardo et al. (2022), the dataset paper, define a bankruptcy event in the year following the final fiscal-year observation of a failed company. Earlier observations of the same company are therefore negative for the one-year-ahead task.

The CSV used here is the correctly aligned Kaggle copy. Its checksum is recorded in the dataset README.

In [ ]:
american = pd.read_csv(AMERICAN_FILE)

required_columns = {
    "company_name",
    "status_label",
    "year",
    "X1",
    "X4",
    "X10",
    "X14",
    "X17",
}
missing_columns = required_columns.difference(american.columns)
if missing_columns:
    raise ValueError(f"Missing American dataset columns: {sorted(missing_columns)}")

if len(american) != 78_682:
    raise ValueError(f"Expected 78,682 rows, found {len(american):,}.")

last_year = american.groupby("company_name")["year"].transform("max")
american_event = american["status_label"].eq("failed") & american["year"].eq(last_year)

american_equity_to_assets = (american["X10"] - american["X17"]) / american["X10"]
american_ebitda_to_assets = american["X4"] / american["X10"]
american_current_ratio = american["X1"] / american["X14"]

american_criterion = (
    american_equity_to_assets.lt(0)
    & american_ebitda_to_assets.lt(0)
    & american_current_ratio.lt(CURRENT_RATIO_THRESHOLD)
)

american_results = pd.DataFrame(
    [binary_metrics(american_event, american_criterion)],
    index=["one_year_ahead"],
)
display_results(american_results)

## Interpretation

The relevant claims are association and enrichment, not equivalence. A high odds ratio or lift shows that the criterion isolates companies at elevated bankruptcy risk. Recall is expected to remain low because legal bankruptcy can occur without all three accounting conditions being present simultaneously, while some severely distressed companies recover or restructure without filing.

The Polish dataset is economically closest to V4FinBench, supplies the three ratios directly, and shows a strong association (44.5% one-year bankruptcy rate among criterion-positive firms versus 6.9% overall). The American dataset provides the cleaner legal endpoint through Chapter 7 and Chapter 11 filings. The association there is weaker but still highly significant, as expected for listed companies that can sustain negative equity and negative EBITDA for extended periods. Note that the American sample is a panel with repeated firm-years, so the Fisher p-value overstates the effective sample size. A firm-clustered logistic regression leaves the odds ratio unchanged (3.72, 95% CI 2.96–4.69, p < 1e-28). Together, the results support interpreting the criterion as a conservative severe-distress indicator associated with subsequent bankruptcy.